In [ ]:
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
college_cr = pd.read_csv("CollegeCr-1.csv")
college_cr['Neighborhood'] = "College Cr"
edwards = pd.read_csv("Edwards-1.csv")
edwards['Neighborhood'] = "Edwards"
old_town = pd.read_csv("OldTown-1.csv")
old_town['Neighborhood'] = "Old Town"

#missing variables: Street, LandContour, LandSlope, YearRemodAdd, BsmtExposure, BsmtUnfSF, Kitchen
#drop utilities as all rows have the same values across the three neighborhoods
new_column_order = ['Neighborhood','Foundation','GrLivArea','PavedDrive','RoofStyle','Fireplaces','LotInfo','WoodDeckSF','BldgType',
                    'HalfBath','YearBuilt','FullBath','KitchenQual','BsmtFinType1','CentralAir','SaleType','Electrical','BsmtFinSF1',
                    'Exterior','OpenPorchSF','BsmtQual','BsmtCond','RoofMatl','HeatingQC','BedroomAbvGr','YrSold','Heating','OverallQual',
                    'GarageType','HouseStyle','OverallCond','TotRmsAbvGrd','Utilities','SalePrice']
college_cr_reorder = college_cr[new_column_order]
edwards_reorder = edwards[new_column_order]
old_town_reorder = old_town[new_column_order]

#union all neighborhoods
all_features = pd.concat([college_cr_reorder,edwards_reorder,old_town_reorder],axis=0)

#split LotInfo and Exterior
split_lot = all_features['LotInfo'].str.split(';', expand=True)
split_exterior = all_features['Exterior'].str.split(';', expand=True)

all_features[['LotConfig','LotShape','LotArea','LotFrontage']] = split_lot
all_features[['Exterior1st','ExterQual','ExterCond']] = split_exterior

all_features = all_features.drop(['LotInfo','Exterior','Utilities'], axis=1)
all_features = all_features.drop_duplicates()

#convert object columns to numeric
all_features['LotArea'] = pd.to_numeric(all_features['LotArea'],errors='coerce')
all_features['LotFrontage'] = pd.to_numeric(all_features['LotFrontage'],errors='coerce')

#Update Nulls in BsmtCond, BsmtQual, BsmtFinType1, GarageType
all_features['BsmtCond'] = all_features['BsmtCond'].fillna(0)
all_features['BsmtQual'] = all_features['BsmtQual'].fillna('noBasement')
all_features['BsmtFinType1'] = all_features['BsmtFinType1'].fillna('noBasement')
all_features['GarageType'] = all_features['GarageType'].fillna('noGarage')

# Replace any categorical variable that is binary with 1 and 0
all_features['RoofStyle'] = all_features['RoofStyle'].replace({'Gable': 1, 'notGable': 0})
all_features['RoofMatl'] = all_features['RoofMatl'].replace({'CompShg': 1, 'notCompShg': 0})
all_features['ExterCond'] = all_features['ExterCond'].replace({'Gd': 1, 'TA': 0})
all_features['ExterQual'] = all_features['ExterQual'].replace({'Gd': 1, 'TA': 0})
all_features['BsmtCond'] = all_features['BsmtCond'].replace({'TA': 1})
all_features['CentralAir'] = all_features['CentralAir'].replace({'Y': 1, 'N': 0})
all_features['Electrical'] = all_features['Electrical'].replace({'SBrkr': 1, 'Fuse': 0})
all_features['SaleType'] = all_features['SaleType'].replace({'WD': 1, 'notWD': 0})

#Create No. of Years since Built Field
all_features = all_features.assign(YrsSinceBuilt=all_features['YrSold'] - all_features['YearBuilt'])
all_features = all_features.drop(['YearBuilt','YrSold'], axis=1)

#Reset Index
all_features = all_features.reset_index(drop=True)

#Assume errors in data scrape
#Remove row with GrLivArea 5642 and BsmtFinSF1 5644 and LotFrontage 313 - far outside of the IQR for these fields
all_features = all_features.drop(all_features[(all_features['GrLivArea']==5642)&(all_features['BsmtFinSF1']==5644)&(all_features['LotFrontage']==313)].index)
#Remove row with GrLivArea 3608 and LotArea 22950 and Sale Price 475000 - far outside of the IQR for these fields
all_features = all_features.drop(all_features[(all_features['GrLivArea']==3608)&(all_features['LotArea']==22950)&(all_features['SalePrice']==475000)].index)
#Remove row where year is out of normal date range and YrsSinceBuilt is negative
all_features = all_features.drop(all_features[all_features['YrsSinceBuilt']==-3].index)

#Save lot frontage averages by neighborhood
lot_frontage_means = all_features.groupby('Neighborhood')['LotFrontage'].mean()
lot_frontage_college_cr = lot_frontage_means['College Cr']
lot_frontage_edwards = lot_frontage_means['Edwards']
lot_frontage_old_town = lot_frontage_means['Old Town']

#Impute Mean LotFrontage by Neighborhood when Null
all_features['LotFrontage'] = all_features['LotFrontage'].fillna(all_features.groupby('Neighborhood')['LotFrontage'].transform('mean'))

#Create Dummy Variables
all_features = pd.get_dummies(all_features, columns=['Neighborhood', 'Foundation', 'PavedDrive','BldgType', 'KitchenQual', 'BsmtFinType1','BsmtQual','HeatingQC',
                                                     'Heating','GarageType', 'HouseStyle', 'LotConfig', 'LotShape', 'Exterior1st'], drop_first=True, dtype=int)

#drop Heating_Wall and Heating_Grav as they don't exist in Test Sets
all_features = all_features.drop(['Heating_Grav','Heating_Wall'],axis=1)

#273 rows remain in train set after union, dropping duplicates, and dropping outlier rows

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Calculate the correlation matrix
correlation_matrix = all_features.corr()

upper_tri = correlation_matrix.where(np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool))

# Filter for correlations greater than 0.7 (absolute value)
high_correlations = upper_tri[abs(upper_tri) > 0.7]

# Drop NaN values and unstack to get a Series of highly correlated pairs
result = high_correlations.stack().sort_values(ascending=False)

print("Highly correlated pairs (absolute correlation > 0.7):")
print(result)

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
# Calculate VIF for each feature
vif = pd.DataFrame()
vif["Feature"] = all_features.columns
vif["VIF"] = [variance_inflation_factor(all_features.values, i) for i in range(all_features.shape[1])]

# Display VIF
print("Variance Inflation Factor (VIF)")
print(vif)

In [ ]:
#Drop Completely Correlated Variable
all_features = all_features.drop(['BsmtFinType1_noBasement'],axis=1)

#Note:  VIF and Correlations show multicollinearity in variables.  Consider Random Forest/Ridge/Lasso/Elastic Net for Models
#These are good for multicollinearity between predictor variables and reduce overfitting

In [ ]:
from math import log1p
#Transform Sales Price Prior to Model Fitting
all_features['SalePriceLog1p'] = all_features['SalePrice'].apply(log1p)

all_features = all_features.drop('SalePrice',axis=1)

In [ ]:
!pip install shortuuid

In [ ]:
#Union Test Sets
college_cr_test = pd.read_csv("CollegeCr-1.test.csv")
college_cr_test['Neighborhood'] = "College Cr"
college_cr_test['SalePriceLog1p'] = np.nan
edwards_test = pd.read_csv("Edwards-1.test.csv")
edwards_test['Neighborhood'] = "Edwards"
edwards_test['SalePriceLog1p'] = np.nan
old_town_test = pd.read_csv("OldTown-1.test.csv")
old_town_test['Neighborhood'] = "Old Town"
old_town_test['SalePriceLog1p'] = np.nan

#missing variables: Street, LandContour, LandSlope, YearRemodAdd, BsmtExposure, BsmtUnfSF, Kitchen
#drop utilities as all rows have the same values across the three neighborhoods
new_column_order = ['uniqueID','Neighborhood','Foundation','GrLivArea','PavedDrive','RoofStyle','Fireplaces','LotInfo','WoodDeckSF','BldgType',
                    'HalfBath','YearBuilt','FullBath','KitchenQual','BsmtFinType1','CentralAir','SaleType','Electrical','BsmtFinSF1',
                    'Exterior','OpenPorchSF','BsmtQual','BsmtCond','RoofMatl','HeatingQC','BedroomAbvGr','YrSold','Heating','OverallQual',
                    'GarageType','HouseStyle','OverallCond','TotRmsAbvGrd','Utilities','SalePriceLog1p']
college_cr_test_reorder = college_cr_test[new_column_order]
edwards_test_reorder = edwards_test[new_column_order]
old_town_test_reorder = old_town_test[new_column_order]

#union all test sets
test_features = pd.concat([college_cr_test_reorder,edwards_test_reorder,old_town_test_reorder],axis=0)

#split LotInfo and Exterior
split_lot = test_features['LotInfo'].str.split(';', expand=True)
split_exterior = test_features['Exterior'].str.split(';', expand=True)

test_features[['LotConfig','LotShape','LotArea','LotFrontage']] = split_lot
test_features[['Exterior1st','ExterQual','ExterCond']] = split_exterior

test_features = test_features.drop(['LotInfo','Exterior','Utilities'], axis=1)
test_features = test_features.drop_duplicates()

#convert object columns to numeric
test_features['LotArea'] = pd.to_numeric(test_features['LotArea'],errors='coerce')
test_features['LotFrontage'] = pd.to_numeric(test_features['LotFrontage'],errors='coerce')

#Update Nulls in BsmtCond, BsmtQual, BsmtFinType1, GarageType
test_features['BsmtCond'] = test_features['BsmtCond'].fillna(0)
test_features['BsmtQual'] = test_features['BsmtQual'].fillna('noBasement')
test_features['BsmtFinType1'] = test_features['BsmtFinType1'].fillna('noBasement')
test_features['GarageType'] = test_features['GarageType'].fillna('noGarage')

# Replace any categorical variable that is binary with 1 and 0
test_features['RoofStyle'] = test_features['RoofStyle'].replace({'Gable': 1, 'notGable': 0})
test_features['RoofMatl'] = test_features['RoofMatl'].replace({'CompShg': 1, 'notCompShg': 0})
test_features['ExterCond'] = test_features['ExterCond'].replace({'Gd': 1, 'TA': 0})
test_features['ExterQual'] = test_features['ExterQual'].replace({'Gd': 1, 'TA': 0})
test_features['BsmtCond'] = test_features['BsmtCond'].replace({'TA': 1})
test_features['CentralAir'] = test_features['CentralAir'].replace({'Y': 1, 'N': 0})
test_features['Electrical'] = test_features['Electrical'].replace({'SBrkr': 1, 'Fuse': 0})
test_features['SaleType'] = test_features['SaleType'].replace({'WD': 1, 'notWD': 0})

#Create No. of Years since Built Field
test_features = test_features.assign(YrsSinceBuilt=test_features['YrSold'] - test_features['YearBuilt'])
test_features = test_features.drop(['YearBuilt','YrSold'], axis=1)

test_features.loc[(test_features['LotFrontage'].isna())&(test_features['Neighborhood']=='College Cr')] = test_features.loc[(test_features['LotFrontage'].isna())&(test_features['Neighborhood']=='College Cr')].fillna(lot_frontage_college_cr)
test_features.loc[(test_features['LotFrontage'].isna())&(test_features['Neighborhood']=='Edwards')] = test_features.loc[(test_features['LotFrontage'].isna())&(test_features['Neighborhood']=='Edwards')].fillna(lot_frontage_edwards)
test_features.loc[(test_features['LotFrontage'].isna())&(test_features['Neighborhood']=='Old Town')] = test_features.loc[(test_features['LotFrontage'].isna())&(test_features['Neighborhood']=='Old Town')].fillna(lot_frontage_old_town)

#Make all Sales Prices null
test_features['SalePriceLog1p'] = pd.NA

#Create Dummy Variables
test_features = pd.get_dummies(test_features, columns=['Neighborhood', 'Foundation', 'PavedDrive','BldgType', 'KitchenQual', 'BsmtFinType1','BsmtQual','HeatingQC',
                                                     'Heating','GarageType', 'HouseStyle', 'LotConfig', 'LotShape', 'Exterior1st'], drop_first=True, dtype=int)

#Drop Completely Correlated Variable
test_features = test_features.drop(['BsmtFinType1_noBasement'],axis=1)

test_features2 = test_features.drop(['uniqueID'],axis=1)


#45 rows in test set

#union train and test sets
train_test_features = pd.concat([all_features,test_features2])

#Standardize all numeric, non binary variables
#'GrLivArea','OpenPorchSF','YrsSinceBuilt','BsmtFinSF1','WoodDeckSF','LotArea','LotFrontage','FullBath','BedroomAbvGr','OverallQual','OverallCond','TotRmsAbvGrd','HalfBath'

mean_GrlivArea = train_test_features['GrLivArea'].mean()
std_GrlivArea = train_test_features['GrLivArea'].std()
mean_OpenPorchSF = train_test_features['OpenPorchSF'].mean()
std_OpenPorchSF = train_test_features['OpenPorchSF'].std()
mean_YrsSinceBuilt = train_test_features['YrsSinceBuilt'].mean()
std_YrsSinceBuilt = train_test_features['YrsSinceBuilt'].std()
mean_BsmtFinSF1 = train_test_features['BsmtFinSF1'].mean()
std_BsmtFinSF1 = train_test_features['BsmtFinSF1'].std()
mean_WoodDeckSF = train_test_features['WoodDeckSF'].mean()
std_WoodDeckSF = train_test_features['WoodDeckSF'].std()
mean_LotArea = train_test_features['LotArea'].mean()
std_LotArea = train_test_features['LotArea'].std()
mean_LotFrontage = train_test_features['LotFrontage'].mean()
std_LotFrontage = train_test_features['LotFrontage'].std()
mean_FullBath = train_test_features['FullBath'].mean()
std_FullBath = train_test_features['FullBath'].std()
mean_BedroomAbvGr = train_test_features['BedroomAbvGr'].mean()
std_BedroomAbvGr = train_test_features['BedroomAbvGr'].std()
mean_OverallQual = train_test_features['OverallQual'].mean()
std_OverallQual = train_test_features['OverallQual'].std()
mean_OverallCond = train_test_features['OverallCond'].mean()
std_OverallCond = train_test_features['OverallCond'].std()
mean_TotRmsAbvGrd = train_test_features['TotRmsAbvGrd'].mean()
std_TotRmsAbvGrd = train_test_features['TotRmsAbvGrd'].std()
mean_HalfBath = train_test_features['HalfBath'].mean()
std_HalfBath = train_test_features['HalfBath'].std()

train_test_features['GrLivArea'] = (train_test_features['GrLivArea'] - mean_GrlivArea)/std_GrlivArea
train_test_features['OpenPorchSF'] = (train_test_features['OpenPorchSF'] - mean_OpenPorchSF)/std_OpenPorchSF
train_test_features['YrsSinceBuilt'] = (train_test_features['YrsSinceBuilt'] - mean_YrsSinceBuilt)/std_YrsSinceBuilt
train_test_features['BsmtFinSF1'] = (train_test_features['BsmtFinSF1'] - mean_BsmtFinSF1)/std_BsmtFinSF1
train_test_features['WoodDeckSF'] = (train_test_features['WoodDeckSF'] - mean_WoodDeckSF)/std_WoodDeckSF
train_test_features['LotArea'] = (train_test_features['LotArea'] - mean_LotArea)/std_LotArea
train_test_features['LotFrontage'] = (train_test_features['LotFrontage'] - mean_LotFrontage)/std_LotFrontage
train_test_features['FullBath'] = (train_test_features['FullBath'] - mean_FullBath)/std_FullBath
train_test_features['BedroomAbvGr'] = (train_test_features['BedroomAbvGr'] - mean_BedroomAbvGr)/std_BedroomAbvGr
train_test_features['OverallQual'] = (train_test_features['OverallQual'] - mean_OverallQual)/std_OverallQual
train_test_features['OverallCond'] = (train_test_features['OverallCond'] - mean_OverallCond)/std_OverallCond
train_test_features['TotRmsAbvGrd'] = (train_test_features['TotRmsAbvGrd'] - mean_TotRmsAbvGrd)/std_TotRmsAbvGrd
train_test_features['HalfBath'] = (train_test_features['HalfBath'] - mean_HalfBath)/std_HalfBath

#Create Housing ID

import shortuuid
#create a unique id
train_test_features['housing_id'] = train_test_features.apply(lambda _: shortuuid.ShortUUID().random(length=5), axis=1)

#Reset Index
train_test_features = train_test_features.reset_index(drop=True)

#Pull housing id and sale price to front
cols_to_move_front = ['housing_id','SalePriceLog1p']
remaining_cols_front = [col for col in train_test_features.columns if col not in cols_to_move_front]
new_order_front = cols_to_move_front + remaining_cols_front
train_test_features = train_test_features[new_order_front]

#340 rows overall

In [ ]:
(0.581870*std_GrlivArea)+mean_GrlivArea

In [ ]:
(-0.695499*std_GrlivArea)+mean_GrlivArea

In [ ]:
(-0.960846*std_GrlivArea)+mean_GrlivArea

In [ ]:
X_test.tail(10)

In [ ]:
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import RidgeCV, Ridge, LassoCV, Lasso, ElasticNet, ElasticNetCV
from sklearn.metrics import root_mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import LeaveOneOut
from sklearn.model_selection import KFold
from sklearn.feature_selection import RFECV
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

# Split the dataset into training and test sets
# Create the full training set where Sale Price is not null
train_mask = train_test_features['SalePriceLog1p'].notna()
train_set = train_test_features[train_mask]

test_mask = train_test_features['SalePriceLog1p'].isna()
test_set = train_test_features[test_mask]

#Select or Drop Fields to be Used in training/testing sets
X_main = train_set.drop(['SalePriceLog1p','housing_id'],axis=1)
y_main = train_set['SalePriceLog1p']

X_test = test_set.drop(['SalePriceLog1p','housing_id'],axis=1)

# Random Forest Variable Selection
# tested n estimators 100, 150, 200
# tested min sample split 2, 3, 5
# tested max depth 5, 7, 10, 11
rf_regressor = RandomForestRegressor(n_estimators=149,min_samples_split=2,max_depth=11,random_state=42,n_jobs=-1)

cv_strategy = KFold(n_splits=10, shuffle=True, random_state=42)
rfecv = RFECV(estimator=rf_regressor, step=1, cv=cv_strategy, scoring='r2', n_jobs=-1)


rfecv = rfecv.fit(X_main, y_main)
optimal_features_index = rfecv.n_features_
print(f"Optimal number of Random Forest features: {rfecv.n_features_}")
selected_features = X_main.columns[rfecv.support_]
print(f"Selected Random Forest features: {selected_features.tolist()}")
print(f"Best RMSE after RFECV: {rfecv.cv_results_["mean_test_score"][optimal_features_index]}")

# cv_scores = rfecv.cv_results_["mean_test_score"] * -1
# print(f"Best STD after RFECV: {rfecv.cv_results_['std_test_score'][optimal_features_index]}")

# import matplotlib.pyplot as plt

# plt.figure(figsize=(10, 6))
# plt.xlabel("Number of features selected")
# plt.ylabel("Cross Validation Score (RMSE)")
# plt.plot(range(1, len(cv_scores) + 1), cv_scores)
# plt.title("Random Forest Recursive Feature Elimination with Cross-Validation (RMSE)")
# plt.show()

#depth 11, 2 splits, estimators 149: 0.14174627344814741

In [ ]:
rf_model = RandomForestRegressor(n_estimators=149,min_samples_split=2,max_depth=11,random_state=42,n_jobs=-1)

rmse_scores = cross_val_score(rf_model, X_main, y_main,
                             scoring="neg_root_mean_squared_error", cv=10)
rmse_scores = -rmse_scores

print("Mean RMSE:", rmse_scores.mean())
print("Standard deviation of RMSE:", rmse_scores.std())

rmse_scores = cross_val_score(rf_model, X_main, y_main,
                             scoring="r2", cv=10)

print("Mean r^2:", rmse_scores.mean())

In [ ]:
rfAdjR = 1 - (((1 - 0.7200076416180912) * (273 - 1)) / (273 - 31 - 1))
rfAdjR

In [ ]:
rfAdjR = 1 - (((1 - 0.8409133950752811) * (273 - 1)) / (273 - 31 - 1))
rfAdjR

In [ ]:
if hasattr(rfecv.estimator_, 'feature_importances_'):
    importances = rfecv.estimator_.feature_importances_
elif hasattr(rfecv.estimator_, 'coef_'):
    # For linear models, use absolute coefficients for importance
    importances = np.abs(rfecv.estimator_.coef_)
    # Handle multi-class coef_ if necessary (e.g., average across classes)
    if importances.ndim > 1:
        importances = np.mean(importances, axis=0)
else:
    raise AttributeError("Estimator does not have feature_importances_ or coef_ attribute")

feature_importances = pd.Series(importances, index=selected_features).sort_values(ascending=False)

# 4. Plot the feature importances
plt.figure(figsize=(10, 5))
feature_importances.plot(kind='barh',width=0.8)
plt.xlabel('Absolute Coefficient Value (Importance)')
plt.ylabel('Feature')
plt.title('Random Forest Feature Importance')
plt.gca().invert_yaxis() # To display the most important feature at the top
plt.show()

In [ ]:
# Ridge Variable Selection
# tried alphas 0.1, 1, 10, 20, 30, 15

cv = KFold(n_splits=10, shuffle=True, random_state=42)
rfecv = RFECV(estimator=Ridge(alpha=15), step=1, cv=cv, scoring='neg_root_mean_squared_error')

# Fit RFECV to select features
rfecv = rfecv.fit(X_main, y_main)

print(f"Optimal number of Ridge features: {rfecv.n_features_}")
selected_features = X_main.columns[rfecv.support_]
print(f"Selected Ridge features: {selected_features.tolist()}")
print(f"Best RMSE after RFECV: {rfecv.cv_results_["mean_test_score"] * -1}")

cv_scores = rfecv.cv_results_["mean_test_score"] * -1

print(f"Best STD after RFECV: {rfecv.cv_results_['std_test_score']}")

# import matplotlib.pyplot as plt

# plt.figure(figsize=(10, 6))
# plt.xlabel("Number of features selected")
# plt.ylabel("Cross Validation Score (RMSE)")
# plt.plot(range(1, len(cv_scores) + 1), cv_scores)
# plt.title("Ridge Recursive Feature Elimination with Cross-Validation (RMSE)")
# plt.show()

In [ ]:
RidgeAdjR = 1 - (((1 - 0.87836751) * (273 - 1)) / (273 - 18 - 1))
RidgeAdjR

In [ ]:
if hasattr(rfecv.estimator_, 'feature_importances_'):
    importances = rfecv.estimator_.feature_importances_
elif hasattr(rfecv.estimator_, 'coef_'):
    # For linear models, use absolute coefficients for importance
    importances = np.abs(rfecv.estimator_.coef_)
    # Handle multi-class coef_ if necessary (e.g., average across classes)
    if importances.ndim > 1:
        importances = np.mean(importances, axis=0)
else:
    raise AttributeError("Estimator does not have feature_importances_ or coef_ attribute")

feature_importances = pd.Series(importances, index=selected_features).sort_values(ascending=False)

# 4. Plot the feature importances
plt.figure(figsize=(10, 5))
feature_importances.plot(kind='barh',width=0.8)
plt.xlabel('Absolute Coefficient Value (Importance)')
plt.ylabel('Feature')
plt.title('Ridge Feature Importance')
plt.gca().invert_yaxis() # To display the most important feature at the top
plt.show()

In [ ]:
# Lasso Regression
lasso_cv_model = LassoCV(cv=10, random_state=42)

lasso_cv_model.fit(X_main, y_main)

optimal_alpha = lasso_cv_model.alpha_
print(f"Optimal alpha found by LassoCV: {optimal_alpha}")

coefficients = lasso_cv_model.coef_
selected_features = np.sum(coefficients != 0)
selected_features_indices = np.where(coefficients != 0)[0]
print(f"Number of selected features: {selected_features}")

feature_names = X_main.columns.tolist()
importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': np.abs(coefficients)})
importance_df = importance_df.sort_values(by='Importance', ascending=False)

# 5. Plot the feature importance
plt.figure(figsize=(10, 12))
plt.barh(importance_df['Feature'], importance_df['Importance'])
plt.xlabel('Absolute Coefficient Value (Importance)')
plt.ylabel('Feature')
plt.title('Lasso Feature Importance')
plt.gca().invert_yaxis() # To display the most important feature at the top
plt.show()

In [ ]:
lasso_cv_model = LassoCV(cv=10, random_state=42)

lasso_cv_model.fit(X_main, y_main)

print(lasso_cv_model.coef_)
print(lasso_cv_model.intercept_)

predictions = lasso_cv_model.predict(X_test)

#expm1

In [ ]:
predictionsBack = np.expm1(predictions)

predictionsDF = pd.DataFrame({'SalePriceTest':predictionsBack})
predictionsDF.SalePriceTest = predictionsDF.SalePriceTest.round(2)
predictionsDF.head(10)

predictionsDF.to_csv('Output.csv')

In [ ]:
predictionsDF.head(4)

In [ ]:
ridge_model = Ridge(alpha=15, random_state=42)

rmse_scores = cross_val_score(ridge_model, X_main, y_main,
                             scoring="neg_root_mean_squared_error", cv=10)
rmse_scores = -rmse_scores

print("Mean RMSE:", rmse_scores.mean())
print("Standard deviation of RMSE:", rmse_scores.std())

rmse_scores = cross_val_score(ridge_model, X_main, y_main,
                             scoring="r2", cv=10)

print("Mean r^2:", rmse_scores.mean())


In [ ]:
RidgeAdjR = 1 - (((1 - 0.7995762162944866) * (273 - 1)) / (273 - 18 - 1))
RidgeAdjR

In [ ]:
lasso_model = Lasso(alpha=0.0038641525863074913, random_state=42)

rmse_scores = cross_val_score(lasso_model, X_main, y_main,
                             scoring="neg_root_mean_squared_error", cv=10)
rmse_scores = -rmse_scores

print("Mean RMSE:", rmse_scores.mean())
print("Standard deviation of RMSE:", rmse_scores.std())

rmse_scores = cross_val_score(lasso_model, X_main, y_main,
                             scoring="r2", cv=10)

print("Mean r^2:", rmse_scores.mean())

In [ ]:
lassoAdjR = 1 - (((1 - 0.8106172250987317) * (273 - 1)) / (273 - 20 - 1))
lassoAdjR

In [ ]:
# Elastic Net Regression

elastic_cv_model = ElasticNetCV(cv=10,
                     random_state=42)
elastic_cv_model.fit(X_main, y_main)

print(f"Best alpha: {elastic_cv_model.alpha_}")
print(f"Best l1_ratio: {elastic_cv_model.l1_ratio_}")

coefficients = elastic_cv_model.coef_
selected_features = np.sum(coefficients != 0)
selected_features_indices = np.where(coefficients != 0)[0]
print(f"Number of selected features: {selected_features}")

feature_names = X_main.columns.tolist()
importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': np.abs(coefficients)})
importance_df = importance_df.sort_values(by='Importance', ascending=False)

# 5. Plot the feature importance
plt.figure(figsize=(10, 12))
plt.barh(importance_df['Feature'], importance_df['Importance'])
plt.xlabel('Absolute Coefficient Value (Importance)')
plt.ylabel('Feature')
plt.title('Elastic Net Feature Importance')
plt.gca().invert_yaxis() # To display the most important feature at the top
plt.show()

In [ ]:
elastic_model = ElasticNet(alpha=0.0077283051726149835, l1_ratio=0.5,random_state=42)

rmse_scores = cross_val_score(elastic_model, X_main, y_main,
                             scoring="neg_root_mean_squared_error", cv=10)
rmse_scores = -rmse_scores

print("Mean RMSE:", rmse_scores.mean())
print("Standard deviation of RMSE:", rmse_scores.std())

rmse_scores = cross_val_score(elastic_model, X_main, y_main,
                             scoring="r2", cv=10)

print("Mean r^2:", rmse_scores.mean())

In [ ]:
#Adjusted R^2

elasticAdjR = 1 - (((1 - 0.8104495207380256) * (273 - 1)) / (273 - 20 - 1))
elasticAdjR